# 05 — Đánh giá: retrieval, AP class mới, giữ class cũ

**Input:** output NB1, NB3, NB2, NB4 (version cuối, đủ run). **GPU:** cần. **Output:** `art/eval/`, `art/eval/report.md`.

Final test chỉ dùng ở notebook này. So base classes theo TÊN class trên cùng ảnh test: base model đánh giá với taxonomy base
(bỏ box novel), model mở rộng với taxonomy base + novel; AP từng class tính độc lập theo class.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"                                    # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C
cfg = C.load_config(SMOKE)
print(cfg["art"])

In [ ]:
STAGE = "eval"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE)

In [ ]:
from topicx import data as D, yolo as Y, metrics as M
meta = C.load(C.find(cfg, "splits/meta.json"))
NOVEL, names, base_names = meta["novel"], meta["names"], meta["base_names"]
B = cfg["branches"] + cfg["analysis_branches"]
paths = D.split_paths(cfg)
pub_boxes = pd.read_csv(C.find(cfg, "splits/public_boxes.csv"))
pub_images = pd.read_csv(C.find(cfg, "splits/public_images.csv"))
test = C.read_ids(C.find(cfg, "splits/test_ids.txt"))
E_ = cfg["eval"]; imgsz = cfg["yolo"]["imgsz"]; W = cfg["yolo"]["weights"]
YD = C.SCRATCH / "yolo"
def ydir(name, ids, nm):
    d = D.write_yolo(YD / name, ids, pub_boxes, nm, paths, cfg["img_wh"])
    return D.write_data_yaml(YD / f"{name}.yaml", nm, d, d, d)
y_base, y_full = ydir("test_base", test, base_names), ydir("test_full", test, names)
tod = dict(zip(pub_images.image, pub_images.timeofday))
subsets = {}
for t in ("daytime", "night"):
    ids_t = [i for i in test if tod[i] == t]
    n_nov = int(pub_boxes[pub_boxes.image.isin(set(ids_t)) & (pub_boxes.cls == NOVEL)].shape[0])
    print(t, len(ids_t), "ảnh,", n_nov, "novel instances")
    if n_nov >= E_["min_instances_subset"]:
        subsets[t] = ydir(f"test_{t}", ids_t, names)

In [ ]:
base_ap = Y.eval_ap(C.find(cfg, f"base/run/{W}.pt"), y_base, "test", imgsz, E_["batch"])
base_map = np.nanmean([base_ap.get(c, {}).get("ap50_95", np.nan) for c in base_names])
rows, missing = [], []
for s in cfg["seeds"]:
    for k in cfg["ks"]:
        for b in B:
            run = f"s{s}_k{k}_{b}"
            res = C.find(cfg, f"runs/{run}/result.json", required=False)
            if res is None:
                missing.append(run); continue
            r = C.load(res)
            cache = C.find(cfg, f"eval/{run}.json", required=False)
            if cache is None:
                w = res.parent / f"{W}.pt"
                ap = Y.eval_ap(w, y_full, "test", imgsz, E_["batch"])
                hits = Y.recall_at_conf(w, test, paths, pub_boxes, NOVEL, E_["conf"], E_["iou"], imgsz, E_["batch"])
                hits["size"] = D.size_bucket(hits).to_numpy()
                hits["tod"] = hits.image.map(tod)
                ev = {"ap": ap, "recall": float(hits.hit.mean()),
                      "recall_by_size": hits.groupby("size").hit.mean().to_dict(),
                      "recall_by_tod": hits.groupby("tod").hit.mean().to_dict(),
                      "novel_ap_by_tod": {t: Y.eval_ap(w, y, "test", imgsz, E_["batch"]).get(NOVEL, {}).get("ap50_95") for t, y in subsets.items()}}
                cache = C.dump(ev, C.out(cfg, f"eval/{run}.json"))
            ev = C.load(cache)
            ap = ev["ap"]
            bm = np.nanmean([ap.get(c, {}).get("ap50_95", np.nan) for c in base_names])
            rows.append({**r, "novel_ap50_95": ap.get(NOVEL, {}).get("ap50_95", 0.0), "novel_ap50": ap.get(NOVEL, {}).get("ap50", 0.0),
                         "base_map": bm, "base_map_delta": bm - base_map,
                         "all_map": np.nanmean([ap.get(c, {}).get("ap50_95", np.nan) for c in names]),
                         f"novel_recall@{E_['conf']}": ev["recall"],
                         **{f"recall_{k_}": v for k_, v in ev["recall_by_size"].items()},
                         **{f"novel_ap_{k_}": v for k_, v in ev["novel_ap_by_tod"].items()}})
df = pd.DataFrame(rows)
print("thiếu run:", missing)
df.to_csv(C.out(cfg, "eval/runs.csv"), index=False)
print(f"base model, base mAP50-95 trên test: {base_map:.4f}")

In [ ]:
# Kiểm tra: trong mỗi seed mọi nhánh dùng chung expanded init và replay set
for s, g in df.groupby("seed"):
    assert g.expanded_md5.nunique() == 1, f"seed {s}: expanded init khác nhau"
assert df.replay_sha.nunique() == 1, "replay set khác nhau"
assert (df.groupby(["seed", "k"]).n_images.nunique() == 1).all(), "số ảnh train khác nhau giữa các nhánh"

In [ ]:
metrics_ = ["novel_ap50_95", "novel_ap50", f"novel_recall@{E_['conf']}", "base_map", "base_map_delta", "all_map",
            "n_novel_images", "n_novel_instances", "n_base_instances_new"]
summ = M.paired_summary(df, metrics_, ref="RANDOM") if "RANDOM" in set(df.branch) else df.groupby(["k", "branch"])[metrics_].agg(["mean", "std"])
ret = pd.read_csv(C.find(cfg, "retrieval/retrieval.csv"))
ret_s = ret.groupby(["k", "branch"])[["precision_at_k", "recall_at_k", "novel_instances", "nn_cos_mean"]].agg(["mean", "std"])
display(ret_s); display(summ)
per_run = df[["seed", "k", "branch"] + metrics_].sort_values(["k", "branch", "seed"])
def md(t):
    try:
        return t.to_markdown()
    except ImportError:
        return "```\n" + t.to_string() + "\n```"
report = f"""# Kết quả — novel = {NOVEL} (smoke={SMOKE})

Base model: base mAP50-95 test = {base_map:.4f}. Seeds = {cfg['seeds']}, K = {cfg['ks']}. Thiếu run: {missing or 'không'}.
Annotation là mô phỏng từ ground truth có sẵn. ORACLE_POSITIVE / RETRIEVAL_MATCHED dùng nhãn ẩn (tham chiếu, không phải phương pháp thực tế).

## Retrieval (pool)
{md(ret_s)}

## Detection (final test), mean/std và hiệu từng cặp so với RANDOM cùng seed
{md(summ)}

## Từng run
{md(per_run.set_index(['k', 'branch', 'seed']))}
"""
C.out(cfg, "eval/report.md").write_text(report, encoding="utf-8")

## Kết luận — điền tay từ bảng trên, trả lời tách 3 câu

1. **Retrieval có thu được nhiều positive hơn random không?** (`precision_at_k`, `novel_instances`, theo K)
2. **Dữ liệu retrieval có cho detection tốt hơn không?** (`d_novel_ap50_95_vs_RANDOM` từng seed, và `base_map_delta` so với REPLAY_ONLY)
3. **Lợi ích có còn khi supervision tương đương không?** (`RETRIEVAL_MATCHED` vs `RANDOM`; nếu không chạy thì ghi là chưa kiểm)

Với 3 seed, báo từng run và mean/std; không tuyên bố ý nghĩa thống kê. COCO pretrained (`yolo11n.pt`) đã có class bus/motorcycle/person/bicycle:
"novel" là mới với base model của tác vụ này, không phải khái niệm mô hình chưa từng thấy.